# 面试问题：Column Parallel 和 Row Parallel Linear 怎样保证与单卡结果一致？

可以直接复述的回答是：第一，Column Parallel 按输出维切权重，各卡算局部输出，最后 all-gather。第二，Row Parallel 按输入维切权重，各卡算局部部分和，最后 all-reduce 再加一次 bias。第三，输入与权重切分必须使用同一边界。第四，不能假设维度一定整除卡数。第五，评估要比较数值误差、单卡权重内存和通信字节。第六，bias 重复相加是 Row Parallel 常见错误。下面用五条推理请求的隐藏向量实现。

## 真实案例：四卡 LLM 输出投影容量验证

五条请求分别代表摘要、问答、代码、客服和批处理任务，每条映射为 8 维隐藏向量；Linear 输出 12 维，切到 4 个逻辑设备。数据和设备均为 CPU 上的确定性教学模拟，不代表真实 GPU 延迟。

In [1]:
import numpy as np  # 使用 NumPy 展示矩阵切分和通信边界
rng = np.random.default_rng(2503)  # 固定权重与隐藏向量保证结果可复现
requests = [  # 定义五条具有批处理语义的推理请求
    {"id": "TP-01", "task": "合同摘要", "tokens": 640},  # 中等长度摘要请求
    {"id": "TP-02", "task": "知识问答", "tokens": 256},  # 短 RAG 请求
    {"id": "TP-03", "task": "代码补全", "tokens": 384},  # 中等代码请求
    {"id": "TP-04", "task": "客服回复", "tokens": 128},  # 短客服请求
    {"id": "TP-05", "task": "离线归纳", "tokens": 1024},  # 长批处理请求
]  # 结束五条推理元数据
hidden = rng.normal(size=(len(requests), 8))  # 为五条请求生成 8 维隐藏状态
weight = rng.normal(size=(8, 12))  # 生成单卡 Linear 的完整权重矩阵
bias = rng.normal(size=(12,))  # 生成只应在归并后添加一次的 bias
print("推理输入：id | task | tokens | hidden_norm")  # 展示批次中的可读请求与数值尺度
for request, vector in zip(requests, hidden):  # 逐条输出五个请求
    print(f"{request['id']} | {request['task']:6} | {request['tokens']:4} | {np.linalg.norm(vector):.3f}")  # 显示隐藏向量范数便于发现异常输入
print("完整 Linear 形状：", hidden.shape, "@", weight.shape, "+", bias.shape)  # 明确矩阵乘法合同


推理输入：id | task | tokens | hidden_norm
TP-01 | 合同摘要   |  640 | 2.463
TP-02 | 知识问答   |  256 | 2.449
TP-03 | 代码补全   |  384 | 3.517
TP-04 | 客服回复   |  128 | 3.933
TP-05 | 离线归纳   | 1024 | 2.740
完整 Linear 形状： (5, 8) @ (8, 12) + (12,)


## Baseline / 基线：单设备保存并计算完整 Linear

单设备直接执行 `X @ W + b`，数值简单但权重和激活都集中在一张卡。它作为两种并行实现的数值 oracle。

In [2]:
baseline_output = hidden @ weight + bias  # 计算单设备完整 Linear 输出
baseline_weight_bytes = weight.nbytes + bias.nbytes  # 统计单设备常驻权重与 bias 字节数
print("单设备输出前两行、前六维：")  # 输出真实数值而非只检查 shape
for request, row in zip(requests[:2], baseline_output[:2]):  # 选择前两条请求展示基线结果
    print(request["id"], np.round(row[:6], 4).tolist())  # 展示后续并行结果需要匹配的数值
print(f"单设备权重内存：{baseline_weight_bytes} bytes")  # 输出容量基线


单设备输出前两行、前六维：
TP-01 [-3.1174, -1.6983, -1.4229, -1.4981, 3.4804, -0.2063]
TP-02 [0.919, -4.587, 3.3853, 1.1116, -0.2904, -0.6335]
单设备权重内存：864 bytes


## 核心实现：输出维 Column 切分与输入维 Row 切分

Column Parallel 拼接四个局部输出后加完整 bias；Row Parallel 对四个部分和做求和，只在归并后加一次 bias。

In [3]:
devices = 4  # 定义四个逻辑设备
column_weights = np.array_split(weight, devices, axis=1)  # 按输出维把完整权重切成四块
column_biases = np.array_split(bias, devices)  # 使用相同输出边界切分 bias
column_locals = [hidden @ shard + local_bias for shard, local_bias in zip(column_weights, column_biases)]  # 各设备独立计算局部输出
column_output = np.concatenate(column_locals, axis=1)  # 模拟 all-gather 拼接完整输出
hidden_shards = np.array_split(hidden, devices, axis=1)  # 按输入维切分每条请求的隐藏状态
row_weights = np.array_split(weight, devices, axis=0)  # 使用同一输入边界切分权重行
row_partials = [local_hidden @ shard for local_hidden, shard in zip(hidden_shards, row_weights)]  # 各设备计算不含 bias 的部分和
row_output = sum(row_partials) + bias  # 模拟 all-reduce 后只添加一次完整 bias
column_error = float(np.max(np.abs(column_output - baseline_output)))  # 计算 Column Parallel 最大数值误差
row_error = float(np.max(np.abs(row_output - baseline_output)))  # 计算 Row Parallel 最大数值误差
print("Column 局部形状：", [array.shape for array in column_locals])  # 展示输出维切分后的每卡结果
print("Row 部分和形状：", [array.shape for array in row_partials])  # 展示输入维切分后的完整形状部分和
print(f"最大误差：column={column_error:.3e}，row={row_error:.3e}")  # 验证两条通信路径与单设备等价


Column 局部形状： [(5, 3), (5, 3), (5, 3), (5, 3)]
Row 部分和形状： [(5, 12), (5, 12), (5, 12), (5, 12)]
最大误差：column=1.776e-15，row=8.882e-16


## 失败案例与修正：整除假设会静默丢列

当输出维从 12 改为 10 时，使用 `chunk = out // devices` 再固定切四块只覆盖 8 列。修正使用显式边界或 `array_split`，允许前两个设备多持有一列。

In [4]:
uneven_weight = rng.normal(size=(8, 10))  # 构造输出维不能被四卡整除的真实边界
uneven_bias = rng.normal(size=(10,))  # 构造对应十维 bias
uneven_reference = hidden @ uneven_weight + uneven_bias  # 计算不均匀切分的单设备 oracle
naive_chunk = uneven_weight.shape[1] // devices  # 错误地使用向下取整固定块宽
naive_shards = [uneven_weight[:, index * naive_chunk:(index + 1) * naive_chunk] for index in range(devices)]  # 固定四块只覆盖前八列
naive_output = np.concatenate([hidden @ shard for shard in naive_shards], axis=1)  # 拼接后得到错误的八维输出
safe_weight_shards = np.array_split(uneven_weight, devices, axis=1)  # 使用不均匀但完整的输出边界
safe_bias_shards = np.array_split(uneven_bias, devices)  # 使用相同边界切分十维 bias
safe_output = np.concatenate([hidden @ shard + local_bias for shard, local_bias in zip(safe_weight_shards, safe_bias_shards)], axis=1)  # 计算完整十维输出
safe_uneven_error = float(np.max(np.abs(safe_output - uneven_reference)))  # 验证修正后与 oracle 一致
print("错误固定切分宽度：", [shard.shape[1] for shard in naive_shards], "输出形状=", naive_output.shape)  # 展示静默丢失两列的失败
print("修正不均匀切分宽度：", [shard.shape[1] for shard in safe_weight_shards], "输出形状=", safe_output.shape)  # 展示完整覆盖十列
print(f"修正后最大误差：{safe_uneven_error:.3e}")  # 量化不均匀切分的数值正确性


错误固定切分宽度： [2, 2, 2, 2] 输出形状= (5, 8)
修正不均匀切分宽度： [3, 3, 2, 2] 输出形状= (5, 10)
修正后最大误差：8.882e-16


## 结果表：内存与通信口径对照

In [5]:
batch = len(requests)  # 使用五条请求作为本次批大小
value_bytes = hidden.dtype.itemsize  # 获取 NumPy float64 的单值字节数
column_per_device_weight = max(shard.nbytes for shard in column_weights)  # 统计 Column Parallel 最大单设备权重字节
row_per_device_weight = max(shard.nbytes for shard in row_weights)  # 统计 Row Parallel 最大单设备权重字节
column_gather_bytes = batch * weight.shape[1] * value_bytes  # 估算 Column Parallel 汇聚完整输出的通信量
row_reduce_bytes = batch * weight.shape[1] * value_bytes  # 估算 Row Parallel 求和完整部分和的逻辑数据量
rows = [("single", baseline_weight_bytes, 0, 0.0), ("column", column_per_device_weight, column_gather_bytes, column_error), ("row", row_per_device_weight, row_reduce_bytes, row_error)]  # 汇总三种执行方案
print("scheme | max_device_weight_bytes | communication_bytes | max_error")  # 输出容量、通信和正确性表
for row in rows:  # 逐方案展示同口径指标
    print(f"{row[0]:6} | {row[1]:7} | {row[2]:5} | {row[3]:.3e}")  # 显示并行以通信换单卡内存的事实


scheme | max_device_weight_bytes | communication_bytes | max_error
single |     864 |     0 | 0.000e+00
column |     192 |   480 | 1.776e-15
row    |     192 |   480 | 8.882e-16


## 结果解读

Column Parallel 的四块局部输出拼接后与单设备逐元素一致；Row Parallel 的四个部分和也一致，因为 bias 只添加一次。两种方案都把最大单设备矩阵权重降到约四分之一，但引入 gather 或 reduce。十维反例说明生产切分必须保存精确边界，不能依赖整除假设。

## 生产边界

真实实现还需 NCCL collective、通信计算重叠、张量布局、异步错误、序列并行、量化 scale 和跨节点拓扑感知。通信字节会受 ring/tree 算法和 dtype 影响，本例只给逻辑数据量。多层网络还需确保上一层输出布局与下一层输入布局兼容。

## 最小回归测试

In [6]:
assert len(requests) >= 5  # 保证并行案例使用至少五条可读推理请求
assert column_error < 1e-10  # 保证 Column Parallel 与单设备输出数值一致
assert row_error < 1e-10  # 保证 Row Parallel 的部分和归并与单设备一致
assert naive_output.shape[1] != uneven_reference.shape[1]  # 保证整除假设反例确实丢失输出列
assert safe_output.shape == uneven_reference.shape  # 保证不均匀切分完整覆盖十维输出
assert safe_uneven_error < 1e-10  # 保证修正后不均匀切分数值正确
